In [ ]:
# This notebook applies the following basic Machine Learning models:
# Logistic Regression, SVM, KNN and Decision Trees
###

In [ ]:
pip install zarr

In [ ]:
# 0. Preparation
###
# Importing libraries
import zarr
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from sklearn.metrics import classification_report, f1_score
from sklearn import linear_model, preprocessing
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
# Mounting GoogleDrive
from google.colab import drive
drive.mount('/content/drive')

# Switch to GPU
!nvidia-smi

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tue Aug 20 21:09:20 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              12W /  70W |      0MiB / 15360MiB |      0%      Default |
|                            

In [ ]:
# Reading data file from GoogleDrive
# Import zarr
data = zarr.open("/content/drive/My Drive/Team Project X-Rays/Dataframes/data_2.1.zarr", mode='r')
target = zarr.open("/content/drive/My Drive/Team Project X-Rays/Dataframes/target_2.1.zarr", mode='r')

# Define data name
df_name = "Baseline 2.0"


In [ ]:
# Explore Data
###

# Check data type is DataFrame
print(type(data))

# Show dimensions
print(data.shape)

# Full size means we have now 90000 columns


<class 'zarr.core.Array'>
(21105, 89401)


In [ ]:
#Calculate the threshold
threshold = 0.8 * data.shape[0]

#Count the number of zeros in each column
zero_counts = np.sum(data[:] == 0, axis=0)

#Identify columns where the number of zeros exceeds the threshold
columns_to_keep = zero_counts <= threshold

#Get the indices of columns to keep
valid_columns = np.where(columns_to_keep)[0]

In [ ]:
#Get the shape of the new array
new_shape = (data.shape[0], len(valid_columns))

#Create a new Zarr array to store the filtered data
data2 = zarr.open("/content/drive/My Drive/Dataframes/data_2.1_2.zarr", mode='w', shape = new_shape, dtype = data.dtype)

#Copy the valid columns to the new array
data2[:] = data[:, valid_columns]

In [ ]:
# Import zarr
data = zarr.open("/content/drive/My Drive/Dataframes/data_2.1_2.zarr", mode='r')

In [ ]:
#Shape no na data
data.shape

(21105, 33369)

In [ ]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size = 0.2, random_state = 123)



In [ ]:
#Initializing the scaler
scaler = StandardScaler()

#Fitting on the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

#Transforming the test data using the same scaler
X_test_scaled = scaler.transform(X_test)

In [ ]:
# PCA: Recuding dimensions
###

# We define n = 0.9 (so < 1) to retain 90 % of explained variance
pca = PCA(n_components = 0.9)

# Now apply the PCA on the training and test set.
# Attention: Make sure that the transformation to both test and train set is on the same N dimensions
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

# Show number of principal components retained
print("Number of components retained:", pca.n_components_)



Number of components retained: 320


In [ ]:
#Checking shape of data after pca
print(X_train.shape)
print(X_test.shape)

(16884, 320)
(4221, 320)


In [ ]:
#Saving PCA results to circumvent RAM issues
np.save('/content/drive/My Drive/X_train_pca_unmasked.npy', X_train)
np.save('/content/drive/My Drive/X_test_pca_unmasked.npy', X_test)

In [ ]:
# 2. Model 1 - Logistic Regression
###
import time
start_time = time.time()

# Instantiate Logistic regression for classification
clf1_name = "Logistic Regression"
clf1 = LogisticRegression(max_iter = 10000, solver = 'lbfgs', C = 1.0)

# Train the model on training data
clf1.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf1.predict(X_test)

# Calc accuracys
clf1_score = clf1.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf1_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model1_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 1: --- %s minutes ---" % model1_time)

# Score and F1-Score
print("The score is:", clf1_score)
print("The mean F1-Score (unweighted) is:", clf1_f1)

# Show Confusion Matrix
#cm1 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
#display(cm1)

# Show classification report
model1_cr = classification_report(y_test, y_pred)
print(model1_cr)


Model 1: --- 0.5533364772796631 minutes ---
The score is: 0.6579009713338072
The mean F1-Score (unweighted) is: 0.5911244272901383
              precision    recall  f1-score   support

         0.0       0.69      0.85      0.76      2103
         1.0       0.47      0.20      0.28       681
         2.0       0.61      0.56      0.58      1165
         3.0       0.76      0.72      0.74       272

    accuracy                           0.66      4221
   macro avg       0.63      0.58      0.59      4221
weighted avg       0.64      0.66      0.63      4221



In [ ]:
# 3. Model 2 - linear SVM
###
import time
start_time = time.time()

# Instantiate SVM
clf2_name = "linear SVM"
clf2 = SVC(gamma = 0.01, kernel = "poly")

# Train the model on training data
clf2.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf2.predict(X_test)

# Calc accuracy
clf2_score = clf2.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf2_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model2_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 2: --- %s minutes ---" % model2_time)

# Score and F1-Score
print("The score is:", clf2_score)
print("The mean F1-Score (unweighted) is:", clf2_f1)

# Show Confusion Matrix
#cm2 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
#display(cm2)

# Show classification report
model2_cr = classification_report(y_test, y_pred)
print(model2_cr)

Model 2: --- 2.332045825322469 minutes ---
The score is: 0.7389244254915897
The mean F1-Score (unweighted) is: 0.7104159301679691
              precision    recall  f1-score   support

         0.0       0.78      0.86      0.82      2103
         1.0       0.54      0.47      0.50       681
         2.0       0.73      0.66      0.69      1165
         3.0       0.84      0.81      0.83       272

    accuracy                           0.74      4221
   macro avg       0.72      0.70      0.71      4221
weighted avg       0.73      0.74      0.73      4221



In [ ]:
# 3. Model 3 - KNN
###
from sklearn import neighbors
import time
start_time = time.time()

# Instantiate classifier
clf3_name = "KNN"
clf3 = KNeighborsClassifier(n_neighbors = 7, metric = 'minkowski')

# Train the model on training data
clf3.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf3.predict(X_test)

# Calc accuracys
clf3_score = clf3.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf3_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model3_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 3: --- %s minutes ---" % model3_time)

# Score and F1-Score
print("The score is:", clf3_score)
print("The mean F1-Score (unweighted) is:", clf3_f1)

# Show Confusion Matrix
#cm3 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
#display(cm3)

# Show classification report
model3_cr = classification_report(y_test, y_pred)
print(model3_cr)

Model 3: --- 0.013975906372070312 minutes ---
The score is: 0.6870409855484483
The mean F1-Score (unweighted) is: 0.6283475507440971
              precision    recall  f1-score   support

         0.0       0.69      0.91      0.78      2103
         1.0       0.53      0.28      0.37       681
         2.0       0.69      0.54      0.60      1165
         3.0       0.94      0.64      0.76       272

    accuracy                           0.69      4221
   macro avg       0.71      0.59      0.63      4221
weighted avg       0.68      0.69      0.67      4221



In [ ]:
# 3. Model 4 - Decision Tree
###
from sklearn.tree import DecisionTreeClassifier
import time
start_time = time.time()

# Instantiate classifier
clf4_name = "Decision Tree"
clf4 = DecisionTreeClassifier(criterion = "entropy", max_depth = 4, random_state = 123)

# Train the model on training data
clf4.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf4.predict(X_test)

# Calc accuracys
clf4_score = clf4.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf4_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model4_time = (time.time() - start_time)/60


In [ ]:
# Show Results
###

# Modelling Time
print("Model 4: --- %s minutes ---" % model4_time)

# Score and F1-Score
print("The score is:", clf4_score)
print("The mean F1-Score (unweighted) is:", clf4_f1)

# Show Confusion Matrix
#cm4 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
#display(cm4)

# Show classification report
model4_cr = classification_report(y_test, y_pred)
print(model4_cr)
# Ideas: Could show most important Features here. But well, there are 4000 pixels...

Model 4: --- 0.09402866760889689 minutes ---
The score is: 0.5733238569059464
The mean F1-Score (unweighted) is: 0.4207116480284398
              precision    recall  f1-score   support

         0.0       0.61      0.84      0.71      2103
         1.0       0.41      0.01      0.02       681
         2.0       0.51      0.42      0.46      1165
         3.0       0.43      0.58      0.49       272

    accuracy                           0.57      4221
   macro avg       0.49      0.46      0.42      4221
weighted avg       0.54      0.57      0.52      4221

